# Bron verwijzingen als hyperlink in de content en pyramide
Dit notebook ondersteunt het verkennen, analyseren van klik functionaliteit binnen AskDelphi.

### Connectie met AskDelphi opzetten

In [11]:
from ask_delphi_api.authentication import AskDelphiClient
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools

client = AskDelphiClient()
client.authenticate()   # pakt automatisch portal code uit .env

project = Project(client)
topic = TopicTools(client, project)

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


### Aanmaken link

In [12]:
import re
import pprint

def create_link(description: str, topicId: str) -> str:
    target = f"target=\"{topicId}\" use=\"default\" view=\"default\""
    thumbnail= "" 
    link = f"link=\"tenant/{client.tenant_id}/project/{client.project_id}/acl/{client.acl_entry_id}/topic/{topicId}/edit"
    link = f"<doppio-link {target} title=\"{description}\" {thumbnail} {link}>{description}</doppio-link>"
    return link

link = create_link("KRB Klantbeeld", "2e04df19-54fa-4341-be5f-0d9c5d1d1635")
pprint.pprint(link)


('<doppio-link target="2e04df19-54fa-4341-be5f-0d9c5d1d1635" use="default" '
 'view="default" title="KRB Klantbeeld"  '
 'link="tenant/0be6d42b-c278-44e6-888e-ba122840d690/project/397296f6-20dd-45cd-8459-250db2725140/acl/4ecd88f2-979b-4fb0-a95d-175d499bc375/topic/2e04df19-54fa-4341-be5f-0d9c5d1d1635/edit>KRB '
 'Klantbeeld</doppio-link>')


### Zoek en vervang tekst door link

In [13]:
def hyperlink_html(description: str, sources: list[dict]) -> str:
    # Langste titel eerst → voorkomt overlappende matches
    for src in sorted(sources, key=lambda d: len(d["titel"]), reverse=True):
        escaped_title = re.escape(src["titel"])

        # Case‑insensitieve regex – we vervangen overal waar de titel voorkomt
        pattern = re.compile(rf"(?i){escaped_title}")
        
        topicId = "2e04df19-54fa-4341-be5f-0d9c5d1d1635"

        link_tag = create_link(escaped_title, topicId)
        # print(f"link_tag : {link_tag}")

        description = pattern.sub(link_tag, description)
        # print(f"description : {description}")

    return description

In [14]:
sources = [{
    "titel": "Kwijtscheldingsformulier Ondernemers",
    "type": "Bronnen",
    "link": "https://www.belastingdienst.nl/wps/wcm/connect/bldcontentnl/themaoverstijgend/programmas_en_formulieren/kwijtschelding_van_belasting_enof_premie_voor_ondernemingen"
}]
text = "Deze tekst gaat over Kwijtscheldingsformulier ondernemers in 2026."

html = hyperlink_html(text, sources)
print(html)

Deze tekst gaat over <doppio-link target="2e04df19-54fa-4341-be5f-0d9c5d1d1635" use="default" view="default" title="Kwijtscheldingsformulier\ Ondernemers"  link="tenant/0be6d42b-c278-44e6-888e-ba122840d690/project/397296f6-20dd-45cd-8459-250db2725140/acl/4ecd88f2-979b-4fb0-a95d-175d499bc375/topic/2e04df19-54fa-4341-be5f-0d9c5d1d1635/edit>Kwijtscheldingsformulier\ Ondernemers</doppio-link> in 2026.


### Voorbeel structuur uitlezen

In [6]:
# import pprint

# # Haal topic-parts op.
# topicId = "1ef44c28-9974-4899-b29a-fe352c9128d6"
# content = topic.get_topic_parts(topicId=topicId)

# # Selecteer part uit topic met daarin de content.
# body_part = None
# groups = content['topicEditorData']['groups']
# for group in groups:
#     for part in group['parts']:
#         if part["partId"] == "body":
#             body_part = part['editors'][0]['value']['richTextEditor']['value']
#             pprint.pp(body_part)